# Phase 6A Diagnostics - D1 Seed Sweep + D2 Unit-of-Observation Probe

Authorized diagnostic extension of Phase 6A baseline reconciliation. This notebook:

- **D1**: sweeps the RF seed over {42, 2025, 7, 123, 2026} for the two methodologically-acceptable anchor-candidate pipelines (`v0median` = median impute + one-hot, no BMI; `v7mean` = mean impute + one-hot, no BMI) on the frozen Phase 6 folds, to measure the OOF ROC-AUC **noise floor**, test whether the mean-imputation effect is seed-robust, and propose an ablation threshold.
- **D2**: runs a read-only near-duplicate probe to inform the dormant unit-of-observation / grouped-CV question.

It trains no new model family, runs no HPO, generates no submissions, and does not start Phase 7. Phase 6A is closed only by a signed `docs/06_validation/phase6a_acceptance.md` (not created here).


## Boundaries

- Authorized scope: **D1 seed sweep** + **D2 read-only near-duplicate probe** only.
- No Phase 7, no HPO, no V8, no model-family comparison, no submissions, no public-leaderboard use.
- Untouched: `logs/experiment_log.csv` (candidate row only), `data/input/`, `notebooks/_official/`, `references/`, `outputs/submissions/`, `.vscode/settings.json`.
- Frozen folds are loaded from the committed Phase 6 file and integrity-checked (row count, labels, SHA-256); never recomputed.
- `School` is used in D2 for duplicate detection only and is never a model feature.
- Pre-write guards refuse to overwrite existing artifacts; the main log is asserted unchanged after writing.
- Integrity gates stop execution if `v0median@42` / `v7mean@42` do not reproduce the accepted Phase 6 / Phase 6A anchors.


In [1]:
from __future__ import annotations

from datetime import datetime
from pathlib import Path
import hashlib
import platform
import subprocess
import sys

import numpy as np
import pandas as pd
import sklearn
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder


# ----------------------------------------------------------------------------
# Configuration and paths
# ----------------------------------------------------------------------------
def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "data" / "input" / "train.csv").exists() and (candidate / "notebooks").exists():
            return candidate
    raise FileNotFoundError("Could not locate repository root containing data/input/train.csv")


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
EXPERIMENT_ID = "phase6a_d1_d2_diagnostics_v1"
PROJECT_SEED = 42
N_SPLITS = 5
SEED_SWEEP = [42, 2025, 7, 123, 2026]
ID_COL = "Id"
TARGET_COL = "Drafted"
ROLE_CATEGORICAL_COLUMNS = ["Player_Type", "Position_Type", "Position"]

# Integrity anchors (accepted Phase 6 / Phase 6A V0 and V7 OOF ROC-AUC).
V0_ANCHOR_OOF = 0.726616
V7_ANCHOR_OOF = 0.802271
FROZEN_FOLDS_SHA256_16 = "96937649526bcadb"  # first 16 hex chars of committed Phase 6 fold file
ANCHOR_TOL = 1e-6

DATA_DIR = PROJECT_ROOT / "data" / "input"
TRAIN_PATH = DATA_DIR / "train.csv"
FROZEN_FOLDS_PATH = PROJECT_ROOT / "outputs" / "folds" / "phase6_rf_sanity_baseline_v1_fold_assignments.csv"
PHASE6_OOF_PATH = PROJECT_ROOT / "outputs" / "oof" / "phase6_rf_sanity_baseline_v1_oof_predictions.csv"
MAIN_LOG_PATH = PROJECT_ROOT / "logs" / "experiment_log.csv"
NOTEBOOK_PATH = PROJECT_ROOT / "notebooks" / "05_phase6a_d1_d2_diagnostics.ipynb"

OUT_OOF = PROJECT_ROOT / "outputs" / "oof"
OUT_VAL = PROJECT_ROOT / "outputs" / "validation"
OUT_REP = PROJECT_ROOT / "outputs" / "reports"
D1_SUMMARY_PATH = OUT_VAL / "phase6a_d1_seed_sweep_summary.csv"
D2_CANDIDATES_PATH = OUT_VAL / "phase6a_d2_duplicate_candidates.csv"
REPORT_PATH = OUT_REP / "phase6a_d1_d2_diagnostics_report.md"
LOG_CANDIDATE_PATH = OUT_REP / "phase6a_d1_d2_experiment_log_candidate.csv"

PIPELINES = {"v0median": "median", "v7mean": "mean"}
OOF_PATHS = {}
for _seed in SEED_SWEEP:
    OOF_PATHS[("v0median", _seed)] = OUT_OOF / f"phase6a_d1_v0median_seed{_seed}_oof_predictions.csv"
    OOF_PATHS[("v7mean", _seed)] = OUT_OOF / f"phase6a_d1_v7mean_seed{_seed}_oof_predictions.csv"
EXPECTED_ARTIFACTS = [D1_SUMMARY_PATH, D2_CANDIDATES_PATH, REPORT_PATH, LOG_CANDIDATE_PATH, *OOF_PATHS.values()]

print(f"Project root: {PROJECT_ROOT}")
print(f"Experiment ID: {EXPERIMENT_ID}")
print(f"Seed sweep: {SEED_SWEEP}")

# Pre-write guard: refuse to overwrite existing artifacts.
existing_artifacts = [str(p.relative_to(PROJECT_ROOT)) for p in EXPECTED_ARTIFACTS if p.exists()]
if existing_artifacts:
    raise FileExistsError("Refusing to overwrite existing D1/D2 artifacts: " + "; ".join(existing_artifacts))

# ----------------------------------------------------------------------------
# Load inputs and assert fold integrity
# ----------------------------------------------------------------------------
for required in [TRAIN_PATH, FROZEN_FOLDS_PATH, PHASE6_OOF_PATH, MAIN_LOG_PATH]:
    if not required.exists():
        raise FileNotFoundError(f"Missing required input: {required}")

train = pd.read_csv(TRAIN_PATH)
frozen_folds = pd.read_csv(FROZEN_FOLDS_PATH)
main_log_before = MAIN_LOG_PATH.read_text(encoding="utf-8")

if list(frozen_folds.columns) != [ID_COL, "fold"]:
    raise ValueError(f"Frozen folds columns must be ['Id', 'fold']; got {list(frozen_folds.columns)}")
if len(frozen_folds) != len(train) or frozen_folds[ID_COL].nunique() != len(train):
    raise ValueError("Frozen folds must contain exactly one row per training Id")
if sorted(frozen_folds["fold"].unique().tolist()) != list(range(N_SPLITS)):
    raise ValueError("Frozen folds must use labels 0..4")
if not frozen_folds[ID_COL].reset_index(drop=True).equals(train[ID_COL].reset_index(drop=True)):
    raise ValueError("Frozen folds must match training Id order")

frozen_sha16 = hashlib.sha256(FROZEN_FOLDS_PATH.read_bytes()).hexdigest()[:16]
if frozen_sha16 != FROZEN_FOLDS_SHA256_16:
    raise ValueError(f"INTEGRITY FAIL: frozen-folds sha256[:16] {frozen_sha16} != expected {FROZEN_FOLDS_SHA256_16}")
print(f"Frozen folds OK: {len(frozen_folds)} rows, labels 0..4, sha256[:16]={frozen_sha16}")

fold_array = frozen_folds["fold"].to_numpy()
y = train[TARGET_COL].astype(int).reset_index(drop=True)


# ----------------------------------------------------------------------------
# Leakage-safe pipeline helpers (mirror the reviewed Phase 6A fold-safe build)
# ----------------------------------------------------------------------------
def make_feature_frame() -> pd.DataFrame:
    feature_df = train.drop(columns=[ID_COL, TARGET_COL, "School"]).copy()
    if ID_COL in feature_df.columns or TARGET_COL in feature_df.columns or "School" in feature_df.columns:
        raise ValueError("Forbidden column leaked into feature frame")
    return feature_df


def make_ohe() -> OneHotEncoder:
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=False)


def make_rf(seed: int) -> RandomForestClassifier:
    return RandomForestClassifier(n_estimators=100, max_depth=5, random_state=seed, n_jobs=-1)


def get_positive_class_proba(estimator, X_valid, positive_label: int = 1) -> np.ndarray:
    model = estimator.named_steps["model"] if isinstance(estimator, Pipeline) else estimator
    classes = getattr(model, "classes_", None)
    if classes is None:
        raise ValueError("Estimator classes_ unavailable after fit")
    matches = np.where(classes == positive_label)[0]
    if len(matches) != 1:
        raise ValueError(f"Positive label {positive_label} not found exactly once in classes_: {classes}")
    proba = estimator.predict_proba(X_valid)[:, int(matches[0])]
    if len(proba) == 0 or pd.isna(proba).any() or not np.isfinite(proba).all() or ((proba < 0) | (proba > 1)).any():
        raise ValueError("Invalid probability vector")
    return proba


def build_pipeline(impute_strategy: str, seed: int):
    X_full = make_feature_frame()
    categorical_columns = [c for c in ROLE_CATEGORICAL_COLUMNS if c in X_full.columns]
    numeric_columns = [c for c in X_full.columns if c not in categorical_columns]
    transformers = []
    if numeric_columns:
        transformers.append(("numeric", SimpleImputer(strategy=impute_strategy), numeric_columns))
    if categorical_columns:
        cat_pipe = Pipeline([("imputer", SimpleImputer(strategy="most_frequent")), ("encoder", make_ohe())])
        transformers.append(("categorical", cat_pipe, categorical_columns))
    preprocessor = ColumnTransformer(transformers=transformers, remainder="drop")
    pipe = Pipeline([("preprocessor", preprocessor), ("model", make_rf(seed))])
    return pipe, X_full


def run_pipeline(impute_strategy: str, seed: int):
    oof_parts = []
    fold_aucs = {}
    for fold in range(N_SPLITS):
        train_mask = fold_array != fold
        valid_mask = fold_array == fold
        pipe, X_full = build_pipeline(impute_strategy, seed)
        pipe.fit(X_full.loc[train_mask], y.loc[train_mask])
        proba = get_positive_class_proba(pipe, X_full.loc[valid_mask], positive_label=1)
        oof_parts.append(
            pd.DataFrame(
                {
                    ID_COL: train.loc[valid_mask, ID_COL].to_numpy(),
                    "fold": fold,
                    "y_true": y.loc[valid_mask].to_numpy(),
                    "y_pred_proba": proba,
                }
            )
        )
        fold_aucs[fold] = float(roc_auc_score(y.loc[valid_mask], proba))
    oof = pd.concat(oof_parts, ignore_index=True).sort_values(ID_COL).reset_index(drop=True)
    for fold in range(N_SPLITS):
        group = oof[oof["fold"] == fold]
        if group["y_true"].nunique() < 2:
            raise ValueError(f"Fold {fold} has only one class")
    oof_auc = float(roc_auc_score(oof["y_true"], oof["y_pred_proba"]))
    return oof, oof_auc, fold_aucs


# ----------------------------------------------------------------------------
# D1 - seed sweep
# ----------------------------------------------------------------------------
records = []
oof_store = {}
for seed in SEED_SWEEP:
    for pname, strategy in PIPELINES.items():
        oof, oof_auc, fold_aucs = run_pipeline(strategy, seed)
        oof_store[(pname, seed)] = (oof, oof_auc, fold_aucs)
        for f in range(N_SPLITS):
            records.append({"pipeline": pname, "seed": seed, "scope": f"fold_{f}", "auc": fold_aucs[f]})
        records.append({"pipeline": pname, "seed": seed, "scope": "oof", "auc": oof_auc})
        print(f"{pname} seed={seed}: OOF={oof_auc:.6f}  folds={[round(fold_aucs[f], 6) for f in range(N_SPLITS)]}")

summary_long = pd.DataFrame(records)

# Integrity gates: seed-42 must reproduce the accepted anchors.
v0_42 = oof_store[("v0median", 42)][1]
v7_42 = oof_store[("v7mean", 42)][1]
print(f"\nIntegrity gate: v0median@42 OOF={v0_42:.6f} (anchor {V0_ANCHOR_OOF}); v7mean@42 OOF={v7_42:.6f} (anchor {V7_ANCHOR_OOF})")
if abs(v0_42 - V0_ANCHOR_OOF) > ANCHOR_TOL:
    raise ValueError(f"INTEGRITY FAIL: v0median@42 OOF {v0_42:.6f} != accepted anchor {V0_ANCHOR_OOF}")
if abs(v7_42 - V7_ANCHOR_OOF) > ANCHOR_TOL:
    raise ValueError(f"INTEGRITY FAIL: v7mean@42 OOF {v7_42:.6f} != accepted anchor {V7_ANCHOR_OOF}")

phase6_oof = pd.read_csv(PHASE6_OOF_PATH).sort_values(ID_COL).reset_index(drop=True)
v0_42_oof = oof_store[("v0median", 42)][0].sort_values(ID_COL).reset_index(drop=True)
max_vec_diff = float(np.max(np.abs(phase6_oof["y_pred_proba"].to_numpy() - v0_42_oof["y_pred_proba"].to_numpy())))
print(f"v0median@42 OOF vector vs accepted Phase 6 OOF: max|diff|={max_vec_diff:.2e}")
if max_vec_diff > 1e-9:
    raise ValueError(f"INTEGRITY FAIL: v0median@42 OOF vector differs from accepted Phase 6 OOF (max|diff|={max_vec_diff:.2e})")
print("All D1 integrity gates passed.")


# ----------------------------------------------------------------------------
# D1 - noise floor, paired deltas, proposed threshold
# ----------------------------------------------------------------------------
def stats(arr) -> dict:
    arr = np.asarray(arr, dtype=float)
    return {
        "mean": float(np.mean(arr)),
        "std_sample": float(np.std(arr, ddof=1)),
        "min": float(np.min(arr)),
        "max": float(np.max(arr)),
        "range": float(np.max(arr) - np.min(arr)),
    }


v0_oofs = np.array([oof_store[("v0median", s)][1] for s in SEED_SWEEP])
v7_oofs = np.array([oof_store[("v7mean", s)][1] for s in SEED_SWEEP])
v0_stats = stats(v0_oofs)
v7_stats = stats(v7_oofs)

paired_rows = []
for s in SEED_SWEEP:
    paired_rows.append({"seed": s, "v0median_oof": oof_store[("v0median", s)][1],
                        "v7mean_oof": oof_store[("v7mean", s)][1],
                        "delta_v7_minus_v0": oof_store[("v7mean", s)][1] - oof_store[("v0median", s)][1]})
paired_oof_df = pd.DataFrame(paired_rows)
paired_stats = stats(paired_oof_df["delta_v7_minus_v0"].to_numpy())

fold_paired_rows = []
for s in SEED_SWEEP:
    fa0 = oof_store[("v0median", s)][2]
    fa7 = oof_store[("v7mean", s)][2]
    for f in range(N_SPLITS):
        fold_paired_rows.append({"seed": s, "fold": f, "v0median": fa0[f], "v7mean": fa7[f], "delta": fa7[f] - fa0[f]})
fold_paired_df = pd.DataFrame(fold_paired_rows)
same_sign_positive = int((fold_paired_df["delta"] > 0).sum())
n_fold_pairs = len(fold_paired_df)

# Per-fold seed-noise: std of each fold's AUC across seeds, per pipeline.
fold_noise_rows = []
for pname in PIPELINES:
    for f in range(N_SPLITS):
        vals = np.array([oof_store[(pname, s)][2][f] for s in SEED_SWEEP])
        fold_noise_rows.append({"pipeline": pname, "fold": f, "fold_auc_std_across_seeds": float(np.std(vals, ddof=1))})
fold_noise_df = pd.DataFrame(fold_noise_rows)

seed_noise_std_oof = max(v0_stats["std_sample"], v7_stats["std_sample"])
THRESHOLD_K = 2.0
THRESHOLD_FLOOR = 0.005
proposed_threshold = max(THRESHOLD_K * seed_noise_std_oof, THRESHOLD_FLOOR)
mean_impute_robust = bool(paired_stats["min"] > THRESHOLD_K * seed_noise_std_oof)

# Proposed minimum slice size (for ratification; final value cross-checked against Phase 6 slice variance).
PROPOSED_MIN_SLICE_N = 50
smallest_fold_pos = int(min(int((train.loc[fold_array == f, TARGET_COL]).sum()) for f in range(N_SPLITS)))
smallest_fold_neg = int(min(int((1 - train.loc[fold_array == f, TARGET_COL]).sum()) for f in range(N_SPLITS)))

print(f"\nD1 noise floor (OOF std across seeds): v0median={v0_stats['std_sample']:.6f}, v7mean={v7_stats['std_sample']:.6f}")
print(f"Paired v7mean - v0median OOF delta: mean={paired_stats['mean']:.6f}, min={paired_stats['min']:.6f}, range={paired_stats['range']:.6f}")
print(f"Same-sign positive fold pairs: {same_sign_positive}/{n_fold_pairs}")
print(f"Proposed ablation threshold (= max({THRESHOLD_K}*seed_std, {THRESHOLD_FLOOR})): {proposed_threshold:.6f}")
print(f"Mean-imputation effect robust vs noise: {mean_impute_robust}")


# ----------------------------------------------------------------------------
# D2 - read-only near-duplicate / unit-of-observation probe (no model, no features)
# ----------------------------------------------------------------------------
d2 = train.copy()
d2["Height_r"] = d2["Height"].round(1)
d2["Weight_r"] = d2["Weight"].round(0)
profile_cols = ["School", "Position", "Player_Type", "Position_Type", "Height_r", "Weight_r"]

cluster_records = []
cluster_id = 0
for _, idx in d2.groupby(profile_cols, dropna=False).groups.items():
    sub = d2.loc[idx]
    if len(sub) >= 2:  # identical profile shared by >= 2 rows = within-CV duplicate risk
        cluster_id += 1
        n_years = int(sub["Year"].nunique())
        folds_spanned = int(frozen_folds.set_index(ID_COL).loc[sub[ID_COL].to_numpy(), "fold"].nunique())
        for _, r in sub.sort_values("Year").iterrows():
            cluster_records.append({
                "cluster_id": cluster_id,
                "cluster_size": int(len(sub)),
                "n_distinct_years": n_years,
                "n_folds_spanned": folds_spanned,
                "cross_year": bool(n_years >= 2),
                ID_COL: int(r[ID_COL]),
                "Year": int(r["Year"]),
                "School": r["School"],
                "Position": r["Position"],
                "Player_Type": r["Player_Type"],
                "Position_Type": r["Position_Type"],
                "Height": float(r["Height"]),
                "Weight": float(r["Weight"]),
                "Drafted": int(r[TARGET_COL]),
            })
d2_candidates = pd.DataFrame(cluster_records)

n_clusters = int(d2_candidates["cluster_id"].nunique()) if len(d2_candidates) else 0
n_rows_in_clusters = int(len(d2_candidates))
frac_in_clusters = n_rows_in_clusters / len(train)
if len(d2_candidates):
    cross_year_clusters = int(d2_candidates.loc[d2_candidates["cross_year"], "cluster_id"].nunique())
    cross_year_rows = int(d2_candidates["cross_year"].sum())
    multifold_clusters = int(d2_candidates.loc[d2_candidates["n_folds_spanned"] >= 2, "cluster_id"].nunique())
    multifold_rows = int((d2_candidates["n_folds_spanned"] >= 2).sum())
else:
    cross_year_clusters = cross_year_rows = multifold_clusters = multifold_rows = 0

ESCALATION_FRAC = 0.02
# Escalation = material share of rows in identical-profile clusters that also span >= 2 folds
# (the configuration that would actually leak identity across CV folds).
d2_escalation = (multifold_rows / len(train)) > ESCALATION_FRAC

print(f"\nD2 probe: {n_clusters} identical-profile clusters covering {n_rows_in_clusters} rows ({frac_in_clusters:.2%} of train)")
print(f"  cross-year clusters={cross_year_clusters} (rows={cross_year_rows}); fold-spanning clusters={multifold_clusters} (rows={multifold_rows})")
print(f"  D2 escalation (fold-spanning rows > {ESCALATION_FRAC:.0%}): {d2_escalation}")


# ----------------------------------------------------------------------------
# Build report, candidate log, and write artifacts
# ----------------------------------------------------------------------------
def get_git_status() -> str:
    try:
        commit = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=PROJECT_ROOT, text=True).strip()
        porcelain = subprocess.check_output(["git", "status", "--short"], cwd=PROJECT_ROOT, text=True).strip()
        return f"{commit} ({'dirty' if porcelain else 'clean'})"
    except Exception as exc:  # noqa: BLE001
        return f"git_status_unavailable: {exc}"


def markdown_table(df: pd.DataFrame, digits: int = 6) -> str:
    if df.empty:
        return "_No rows._"
    columns = [str(c) for c in df.columns]

    def fmt(value):
        if pd.isna(value):
            return ""
        if isinstance(value, (float, np.floating)):
            return f"{float(value):.{digits}f}"
        return str(value).replace("\n", " ").replace("|", "\\|")

    lines = ["| " + " | ".join(columns) + " |", "|" + "|".join(["---"] * len(columns)) + "|"]
    for _, row in df.iterrows():
        lines.append("| " + " | ".join(fmt(row[c]) for c in df.columns) + " |")
    return "\n".join(lines)


sweep_rows = []
for s in SEED_SWEEP:
    for pname in PIPELINES:
        _, oof_auc, fa = oof_store[(pname, s)]
        fmean = float(np.mean([fa[f] for f in range(N_SPLITS)]))
        fstd = float(np.std([fa[f] for f in range(N_SPLITS)], ddof=1))
        sweep_rows.append({"pipeline": pname, "seed": s, "oof_auc": oof_auc, "fold_mean": fmean, "fold_std": fstd})
sweep_df = pd.DataFrame(sweep_rows).sort_values(["pipeline", "seed"]).reset_index(drop=True)

noise_df = pd.DataFrame([
    {"pipeline": "v0median", **{k: v0_stats[k] for k in ["mean", "std_sample", "min", "max", "range"]}},
    {"pipeline": "v7mean", **{k: v7_stats[k] for k in ["mean", "std_sample", "min", "max", "range"]}},
])

d2_examples = d2_candidates.sort_values(["cluster_size", "cluster_id"], ascending=[False, True]).head(20) if len(d2_candidates) else d2_candidates

report = f"""# Phase 6A Diagnostics Report - D1 Seed Sweep + D2 Unit-of-Observation Probe

## Scope

Authorized diagnostic extension of Phase 6A. Strictly limited to:

- D1: RF seed sweep of the two methodologically-acceptable anchor-candidate pipelines on the frozen folds.
- D2: read-only near-duplicate / unit-of-observation probe (no model, no features, no protocol change).

Boundary checks:

- No Phase 7, no HPO, no V8, no model-family comparison, no submissions, no public-leaderboard use.
- `logs/experiment_log.csv` not modified (candidate row only, under `outputs/reports/`).
- `School` used in D2 for duplicate detection only; it is never a model feature.
- Frozen folds loaded from the committed Phase 6 file and integrity-checked; never recomputed.

## Environment

| Item | Value |
|---|---|
| Git status | {get_git_status()} |
| Python | {sys.version.split()[0]} |
| Platform | {platform.platform()} |
| numpy | {np.__version__} |
| pandas | {pd.__version__} |
| scikit-learn | {sklearn.__version__} |

## Integrity gates (all passed)

| Gate | Value |
|---|---|
| Frozen-folds sha256[:16] | {frozen_sha16} (== {FROZEN_FOLDS_SHA256_16}) |
| v0median @ seed 42 OOF | {v0_42:.6f} (anchor {V0_ANCHOR_OOF}) |
| v7mean @ seed 42 OOF | {v7_42:.6f} (anchor {V7_ANCHOR_OOF}) |
| v0median@42 OOF vector vs accepted Phase 6 OOF, max|diff| | {max_vec_diff:.2e} |

## D1 seed sweep (per pipeline x seed)

`v0median` = fold-safe median impute + one-hot, no BMI (incumbent V0 anchor).
`v7mean` = fold-safe mean impute + one-hot, no BMI (clean-upgrade candidate; isolates the dominant factor).

{markdown_table(sweep_df)}

## D1 noise floor (OOF ROC-AUC across {len(SEED_SWEEP)} seeds)

{markdown_table(noise_df)}

Seed-noise std (OOF, max of the two pipelines): **{seed_noise_std_oof:.6f}**.

## D1 paired contrast (v7mean - v0median, same seed)

{markdown_table(paired_oof_df)}

Paired OOF delta: mean **{paired_stats['mean']:.6f}**, min **{paired_stats['min']:.6f}**, range {paired_stats['range']:.6f}.
Same-sign positive fold pairs: **{same_sign_positive}/{n_fold_pairs}**.
Mean-imputation effect exceeds {THRESHOLD_K} x seed-noise std on every seed: **{mean_impute_robust}**.

## D1 per-fold seed noise (std of each fold AUC across seeds)

{markdown_table(fold_noise_df)}

## Proposed ablation threshold (for human ratification - NOT auto-adopted)

Proposed rule for Phase 7 feature-block acceptance:

> A block is accepted only if its OOF ROC-AUC gain over the anchor is **>= {proposed_threshold:.6f}**
> (= max({THRESHOLD_K} x seed_std {seed_noise_std_oof:.6f}, floor {THRESHOLD_FLOOR})),
> **and** the per-fold delta is the same sign in **>= 4/5** folds.

Proposed minimum slice size for diagnostics: **n >= {PROPOSED_MIN_SLICE_N}** (smallest fold has {smallest_fold_pos} positives / {smallest_fold_neg} negatives; final value to be cross-checked against the Phase 6 slice-report variance). These are proposals; the Phase 6A acceptance record ratifies the final numbers.

## D2 unit-of-observation probe (read-only)

Heuristic: rows sharing identical (`School`, `Position`, `Player_Type`, `Position_Type`, Height rounded to 0.1, Weight rounded to 1) are flagged as the same-athlete / duplicate-record candidates. Descriptive only.

| Metric | Value |
|---|---|
| Identical-profile clusters (size >= 2) | {n_clusters} |
| Rows in such clusters | {n_rows_in_clusters} ({frac_in_clusters:.2%} of train) |
| Cross-year clusters | {cross_year_clusters} (rows {cross_year_rows}) |
| Clusters spanning >= 2 CV folds | {multifold_clusters} (rows {multifold_rows}) |
| Escalation threshold (fold-spanning rows) | > {ESCALATION_FRAC:.0%} of train |
| **D2 escalation** | **{d2_escalation}** |

Examples (largest clusters):

{markdown_table(d2_examples)}

Interpretation: identical-profile rows that land in different CV folds would let the same athlete's identity leak across folds, optimistically biasing every absolute AUC. Because all D1/V0-V7 contrasts share the same folds, this bias is common-mode and largely cancels in paired deltas; the decomposition conclusions are unaffected. {"D2 escalation triggered: grouped-CV is now a live protocol question - STOP and obtain a human ruling before Phase 7." if d2_escalation else "D2 escalation NOT triggered: grouped CV remains dormant; unit of observation stays 'Not confirmed yet' pending the Phase 6A acceptance record."}

## Closure status

D1 and D2 are executed and ready for human review. They do not by themselves close Phase 6A: the anchor, encoding policy, BMI disposition, ablation threshold, and minimum slice size are ratified in `docs/06_validation/phase6a_acceptance.md` (not created here).

## Next gate

Do not start Phase 7 until the Phase 6A acceptance record is signed.
"""

candidate_log = pd.DataFrame([{
    "experiment_id": EXPERIMENT_ID,
    "date": datetime.now().strftime("%Y-%m-%d"),
    "phase": "Phase 6A - D1/D2 diagnostics",
    "notebook_or_script": str(NOTEBOOK_PATH.relative_to(PROJECT_ROOT)).replace("\\", "/"),
    "git_commit_or_status": get_git_status(),
    "data_version": "official_data_input_csvs",
    "fold_strategy": "loaded_frozen_phase6_folds; StratifiedKFold(n_splits=5, shuffle=True, random_state=42)",
    "random_seed": "seed sweep [42, 2025, 7, 123, 2026]",
    "feature_block": "phase6a_d1_seed_sweep_and_d2_probe",
    "preprocessing_summary": "D1: fold-safe median/mean impute + one-hot, no BMI; D2: read-only duplicate probe",
    "model_family": "RandomForestClassifier",
    "model_params_summary": "n_estimators=100; max_depth=5; seed swept; no HPO",
    "hpo_status": "not_run",
    "cv_auc_mean": float(v0_stats["mean"]),
    "cv_auc_std": float(v0_stats["std_sample"]),
    "fold_scores": "; ".join(f"{p}@{s}:oof={oof_store[(p, s)][1]:.6f}" for s in SEED_SWEEP for p in PIPELINES),
    "slice_metrics_available": False,
    "leakage_checks_passed": "fold_safe_pipelines; frozen_folds_integrity_ok; main_log_unchanged; no_submission",
    "submission_created": False,
    "submission_path": None,
    "public_lb_score_if_submitted": None,
    "notes": (
        f"D1 seed-noise std(OOF)={seed_noise_std_oof:.6f}; mean-impute paired delta min={paired_stats['min']:.6f} "
        f"(robust={mean_impute_robust}); proposed threshold={proposed_threshold:.6f}; "
        f"D2 fold-spanning rows={multifold_rows} escalation={d2_escalation}"
    ),
    "decision": "ready_for_phase6a_human_review",
}])

# Re-check pre-write guard immediately before writing.
existing_now = [str(p.relative_to(PROJECT_ROOT)) for p in EXPECTED_ARTIFACTS if p.exists()]
if existing_now:
    raise FileExistsError("Refusing to overwrite existing D1/D2 artifacts before write: " + "; ".join(existing_now))

for directory in [OUT_OOF, OUT_VAL, OUT_REP]:
    directory.mkdir(parents=True, exist_ok=True)

for (pname, seed), path in OOF_PATHS.items():
    oof_store[(pname, seed)][0].to_csv(path, index=False)
summary_long.to_csv(D1_SUMMARY_PATH, index=False)
d2_candidates.to_csv(D2_CANDIDATES_PATH, index=False)
REPORT_PATH.write_text(report, encoding="utf-8")
candidate_log.to_csv(LOG_CANDIDATE_PATH, index=False)

main_log_after = MAIN_LOG_PATH.read_text(encoding="utf-8")
if main_log_before != main_log_after:
    raise RuntimeError("logs/experiment_log.csv changed during D1/D2, which is forbidden")

print("\nArtifacts written:")
for p in EXPECTED_ARTIFACTS:
    print(" ", p.relative_to(PROJECT_ROOT), p.exists())
print("\nMain log unchanged:", main_log_before == main_log_after)
print(f"D2 escalation: {d2_escalation}")


Project root: C:\GitHub\reto_Tokio
Experiment ID: phase6a_d1_d2_diagnostics_v1
Seed sweep: [42, 2025, 7, 123, 2026]
Frozen folds OK: 2781 rows, labels 0..4, sha256[:16]=96937649526bcadb


v0median seed=42: OOF=0.726616  folds=[0.690076, 0.751758, 0.761581, 0.704939, 0.737911]


v7mean seed=42: OOF=0.802271  folds=[0.78276, 0.8265, 0.817359, 0.771712, 0.830924]


v0median seed=2025: OOF=0.723812  folds=[0.685666, 0.746303, 0.766674, 0.693537, 0.735899]


v7mean seed=2025: OOF=0.801970  folds=[0.775016, 0.83148, 0.818602, 0.772591, 0.829847]


...

v7mean seed=123: OOF=0.802747  folds=[0.774662, 0.831089, 0.820108, 0.769529, 0.831845]


v0median seed=2026: OOF=0.729133  folds=[0.701255, 0.74599, 0.773123, 0.702452, 0.740094]


v7mean seed=2026: OOF=0.803264  folds=[0.785446, 0.818652, 0.821841, 0.774376, 0.83417]

Integrity gate: v0median@42 OOF=0.726616 (anchor 0.726616); v7mean@42 OOF=0.802271 (anchor 0.802271)
v0median@42 OOF vector vs accepted Phase 6 OOF: max|diff|=4.44e-16
All D1 integrity gates passed.

D1 noise floor (OOF std across seeds): v0median=0.002718, v7mean=0.001097
Paired v7mean - v0median OOF delta: mean=0.076251, min=0.074131, range=0.004028
Same-sign positive fold pairs: 25/25
Proposed ablation threshold (= max(2.0*seed_std, 0.005)): 0.005436
Mean-imputation effect robust vs noise: True



D2 probe: 119 identical-profile clusters covering 252 rows (9.06% of train)
  cross-year clusters=115 (rows=244); fold-spanning clusters=103 (rows=220)
  D2 escalation (fold-spanning rows > 2%): True



Artifacts written:
  outputs\validation\phase6a_d1_seed_sweep_summary.csv True
  outputs\validation\phase6a_d2_duplicate_candidates.csv True
  outputs\reports\phase6a_d1_d2_diagnostics_report.md True
  outputs\reports\phase6a_d1_d2_experiment_log_candidate.csv True
  outputs\oof\phase6a_d1_v0median_seed42_oof_predictions.csv True
  outputs\oof\phase6a_d1_v7mean_seed42_oof_predictions.csv True
  outputs\oof\phase6a_d1_v0median_seed2025_oof_predictions.csv True
  outputs\oof\phase6a_d1_v7mean_seed2025_oof_predictions.csv True
  outputs\oof\phase6a_d1_v0median_seed7_oof_predictions.csv True
  outputs\oof\phase6a_d1_v7mean_seed7_oof_predictions.csv True
  outputs\oof\phase6a_d1_v0median_seed123_oof_predictions.csv True
  outputs\oof\phase6a_d1_v7mean_seed123_oof_predictions.csv True
  outputs\oof\phase6a_d1_v0median_seed2026_oof_predictions.csv True
  outputs\oof\phase6a_d1_v7mean_seed2026_oof_predictions.csv True

Main log unchanged: True
D2 escalation: True
